In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib.ticker import LogLocator, LogFormatter
from matplotlib.ticker import LogLocator, NullFormatter
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from constants import *
from functions import *

In [2]:
cols = [
    "l", "E_e", "t_IC", "t_syn", "t_br", "t_diff", "t_acc",
    "t_cool", "t_adv", "t_cell",
    "t_IC_star", "t_IC_IR",
    "t_esc", "t_tot",
    "t_adi", "t_coul", "t_ion"
]

## Load data

In [3]:
scenarios = ["B0-05-d", "B1-1-d", "B1-1-h", "B1-1-mc", "B2-60-d"]
shocks = ["RS", "FS"]
cells = {
    10: 19,
    45: 80,
    120: 213
}

data_tiempos = {
    c: {
        sc: {}
        for sc in scenarios
    }
    for c in cells
}

for sc in scenarios:
    for shock in shocks:

        file = f"../../../CD_tcoole_{sc}_{shock}.dat"

        df = pd.read_csv(
            file,
            delim_whitespace=True,
            comment="#",
            names=cols)

        for key, lval in cells.items():

            dfi = df.loc[df["l"] == lval].copy()

            dfi["t_IC"] = 1.0 / (
                1.0 / dfi["t_IC_star"] +
                1.0 / dfi["t_IC_IR"]
                )

            data_tiempos[key][sc][shock] = dfi

## Graph

In [4]:
curves = {
    "t_acc":  {"label": r"$t_{\rm acc}$", "ls": "--", "color": cb_colors["black"],
               "alpha" : 0.8, "width": 2.},  
    "t_diff": {"label": r"$t_{\rm diff}$", "ls": "-.", "color": cb_colors["blue"],
               "alpha" : 0.8, "width": 2.},
    "t_syn":  {"label": r"$t_{\rm syn}$", "ls": "-",  "color": cb_colors["green"],
               "alpha" : 0.8, "width": 2.},
    "t_adv": {"label": r"$t_{\rm adv}$","ls": "-.", "color": "cyan",
              "alpha" : 0.8, "width": 2.},
    "t_IC_star": {"label": "", "ls": "-", "color": "purple",
                  "alpha" : 0.8, "width": 1.},
    "t_IC_IR":   {"label": "", "ls": "-", "color": "orange",
                  "alpha" : 0.8, "width": 1.},
    "t_IC":   {"label": r"$t_{\rm IC}$", "ls": "-",  "color": cb_colors["red"],
               "alpha" : 0.8, "width": 2.},
    "t_tot": {"label": r"$t_{\rm tot}$","ls": "-", "color": "magenta",
              "alpha" : 0.8, "width": 2.},
    "t_adi":  {"label": r"$t_{\rm adi}$","ls": ":",  "color": "grey",
               "alpha" : 0.8, "width": 2.},
}

NameError: name 'cb_colors' is not defined

# RS

In [ ]:
#fig.savefig('tiempos_e_HVSs_RS.pdf', bbox_inches='tight', pad_inches = 0.01)

In [ ]:
thetas = [10, 45, 120]

# =========================
# Plot: 2 rows (systems) x 3 columns (angles)
# =========================
fig, ax = plt.subplots(
    5, 3,
    figsize=(0.99*text_width, 5*0.7*text_width/3.),
    sharex=True,
    sharey=False
)

plt.subplots_adjust(wspace=0.0, hspace=0.0)

for i, sc in enumerate(scenarios):
    for j, th in enumerate(thetas):

        axi = ax[i, j]
        d = data_tiempos[th][sc]['FS']

        for key, style in curves.items():

            if key not in d:
                continue

            axi.plot(
                d["E_e"], d[key],
                label=style["label"],
                ls=style["ls"],
                color=style["color"],
                alpha=style["alpha"],
                linewidth=style["width"]
            )
            
        axi.set_xscale("log")
        axi.set_yscale("log")
        axi.set_xlim(1e6, 1e15)

        axi.tick_params(
            axis="both", which="both",
            top=True, direction="in"
        )

        axi.tick_params(
            axis="x", which="both",
            pad=6
        )

        axi.set_xticks([1e9, 1e12])
        axi.set_xticks(
            [1e7, 1e8, 1e10, 1e11, 1e13, 1e14],
            minor=True)
        
        if (i==0):
            axi.set_yticks([1e8, 1e11, 1e14])
            axi.set_yticks([1e7, 1e9, 1e10, 1e12, 1e13, 1e15], minor=True)
            axi.set_ylim(1e6, 1e16)
        elif (i==4):
            axi.set_yticks([1e3, 1e6, 1e9])
            axi.set_yticks([1e2, 1e4, 1e5, 1e7, 1e8, 1e10], minor=True)
            axi.set_ylim(1e1, 1e11)
        else:
            axi.set_yticks([1e7, 1e10, 1e13])
            axi.set_yticks([1e6, 1e8, 1e9, 1e11, 1e13, 1e14], minor=True)
            axi.set_ylim(1e5, 1e15)

        axi.xaxis.set_minor_formatter(NullFormatter())
        axi.yaxis.set_minor_formatter(NullFormatter())

        axi.tick_params(
            axis="y", which="both",
            left=True,
            right=True,
            direction="in"
        )

        if j == 0:
            axi.tick_params(labelleft=True, labelright=False)

        elif j == len(thetas) - 1:
            axi.tick_params(labelleft=False, labelright=False)

        else:
            axi.tick_params(labelleft=False, labelright=False)

        axi.annotate(
            f"{sc} FS",
            xy=(0.9, 0.9),
            xycoords="axes fraction",
            ha="right", va="top",
        )

        axi.annotate(
            rf"$\theta = {th}^\circ$",
            xy=(0.9, 0.75),
            xycoords="axes fraction",
            ha="right", va="top"
        )

# =========================
# Globals
# =========================
sp = fig.subplotpars
xmid = (sp.left + sp.right) / 2
ymid = (sp.bottom + sp.top) / 2

fig.text(
    xmid, sp.bottom - 0.04,
    r'$E_e \, [{\rm eV}]$',
    ha='center', va='center'
)

fig.text(
    sp.left - 0.08, ymid,
    r"$t$ [s]",
    rotation='vertical',
    ha='center', va='center'
)

handles_all = []
labels_all = []

for i in range(5):
    for j in range(3):
        h, l = ax[i, j].get_legend_handles_labels()
        for hi, li in zip(h, l):
            if li not in labels_all:
                handles_all.append(hi)
                labels_all.append(li)

fig.legend(
    handles_all, labels_all,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 0.96),
    handlelength=1.5
)

plt.show()

In [ ]:
#fig.savefig('tiempos_e_HVSs_FS.pdf', bbox_inches='tight', pad_inches = 0.01)